<a href="https://colab.research.google.com/github/ccoyso/Tapia/blob/main/determinaci%C3%B3n_de_cargadores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================
# Colab: Escenarios y selección óptima de pe y fc
# Dimensionamiento de cargadores (Mall Iquitos, 2025)
# ============================================

# 1) Dependencias
import sys, subprocess
def _ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
_ensure("pandas"); _ensure("openpyxl")

import pandas as pd
import numpy as np
import math
from dataclasses import dataclass

# 2) Parámetros base (puedes editarlos)
N_MOTOS_PEAK = 900          # llegadas de motos en 4 h pico
N_MOTOTAXIS_PEAK = 130      # llegadas de mototaxis en 4 h pico
PEAK_SHARE_DAY = 0.70       # el pico (4 h) representa 70% del día (10:00–22:00)

POWER_MOTO_KW      = 2.0    # Modo 3: potencia por toma para moto
POWER_MOTOTAXI_KW  = 3.0    # Modo 3: potencia por toma para mototaxi
SOCKETS_PER_CHARGER = 4
HOURS_PEAK = 4.0
HOURS_DAY  = 12.0

# Operación con EMS centralizado (algoritmo) – valores base
TS_ALG_MIN = 25.0           # duración efectiva de sesión (min)
U_ALG      = 0.92           # utilización efectiva

# Reglas de selección
TARGET_CHARGERS = 3         # meta: operar con 3 cargadores (12 tomas)
STABILITY_REQ   = 0.70      # estabilidad mínima (≥70% de simulaciones con TARGET_CHARGERS)

# Rango de escenarios (grilla pe × fc)
PE_LIST = np.linspace(0.10, 0.35, 6)      # 0.10, 0.15, ..., 0.35
FC_LIST = np.array([0.40, 0.50, 0.60, 0.70])

# 3) Funciones de cálculo
@dataclass
class Inputs:
    N_motos_peak: int
    N_mototaxis_peak: int
    peak_share_day: float
    pe: float
    fc: float
    ts_min: float
    u: float
    power_moto_kw: float
    power_mototaxi_kw: float
    sockets_per_charger: int
    hours_peak: float

def sessions_peak(Nm, Nmt, pe, fc):
    return (Nm + Nmt) * pe * fc

def capacity_per_charger(ts_min, u, sockets=4, hours_peak=4.0):
    return sockets * (hours_peak / (ts_min/60.0)) * u

def chargers_needed(sessions_pico, ts_min, u, sockets=4, hours_peak=4.0):
    cap = capacity_per_charger(ts_min, u, sockets, hours_peak)
    return int(math.ceil(sessions_pico / cap)), cap

def mean_socket_kw(Nm, Nmt, p_moto, p_mtaxi):
    total = Nm + Nmt
    mix_moto = Nm/total if total else 0.0
    mix_mtax = Nmt/total if total else 0.0
    return p_moto*mix_moto + p_mtaxi*mix_mtax

# 4) Evaluación de la grilla (base EMS)
rows = []
Nm, Nmt = N_MOTOS_PEAK, N_MOTOTAXIS_PEAK
s_pico_total = Nm + Nmt
kw_per_socket_mean = mean_socket_kw(Nm, Nmt, POWER_MOTO_KW, POWER_MOTOTAXI_KW)

for pe in PE_LIST:
    for fc in FC_LIST:
        ses_pico = sessions_peak(Nm, Nmt, pe, fc)
        carg_base, cap_carg = chargers_needed(ses_pico, TS_ALG_MIN, U_ALG, SOCKETS_PER_CHARGER, HOURS_PEAK)
        # Headroom con TARGET_CHARGERS (margen de sesiones si instalo la meta)
        cap_target = TARGET_CHARGERS * cap_carg
        headroom_sessions = cap_target - ses_pico
        rows.append({
            "pe": pe, "fc": fc,
            "Sesiones_pico_4h": ses_pico,
            "Capacidad_por_cargador_ses_4h": cap_carg,
            "Cargadores_base": carg_base,
            "Headroom_ses_con_TARGET": headroom_sessions
        })

df_grid = pd.DataFrame(rows)

# 5) Robustez (Monte Carlo) respecto a TS y U
#   Simulamos variaciones realistas: ts ~ U(22, 28) min, u ~ U(0.89, 0.95)
N_MC = 500
def stability_probability(pe, fc, n_mc=N_MC):
    ses_pico = sessions_peak(Nm, Nmt, pe, fc)
    count_ok = 0
    margins = []
    for _ in range(n_mc):
        ts = np.random.uniform(22.0, 28.0)   # min
        uu = np.random.uniform(0.89, 0.95)
        carg, cap_carg = chargers_needed(ses_pico, ts, uu, SOCKETS_PER_CHARGER, HOURS_PEAK)
        if carg <= TARGET_CHARGERS:
            count_ok += 1
        # margen con TARGET_CHARGERS (positivo = cabemos con TARGET)
        margins.append(TARGET_CHARGERS*cap_carg - ses_pico)
    return {
        "prob_target": count_ok / n_mc,
        "margin_p05": float(np.percentile(margins, 5)),    # margen conservador (5%)
        "margin_p50": float(np.percentile(margins, 50)),   # mediana
        "margin_p95": float(np.percentile(margins, 95))    # optimista
    }

rob_rows = []
for pe in PE_LIST:
    for fc in FC_LIST:
        r = stability_probability(pe, fc)
        rob_rows.append({
            "pe": pe, "fc": fc,
            "Prob_3_cargadores": r["prob_target"],
            "Margen_ses_p05": r["margin_p05"],
            "Margen_ses_p50": r["margin_p50"],
            "Margen_ses_p95": r["margin_p95"],
        })
df_rob = pd.DataFrame(rob_rows)

# 6) Selección automática del escenario “más conveniente”
df_merge = df_grid.merge(df_rob, on=["pe","fc"], how="left")
# Condición de conveniencia: estabilidad alta con 3 cargadores
candidatos = df_merge[(df_merge["Prob_3_cargadores"] >= STABILITY_REQ)]
if len(candidatos) == 0:
    # Si nadie cumple estabilidad, tomar los de menor Cargadores_base,
    # y dentro de ellos el de mayor Prob_3_cargadores y mayor pe*fc
    gmin = df_merge["Cargadores_base"].min()
    candidatos = df_merge[df_merge["Cargadores_base"] == gmin].copy()
    candidatos["score"] = candidatos["Prob_3_cargadores"] + 0.001*(candidatos["pe"]*candidatos["fc"])
    ganador = candidatos.sort_values(["score","Headroom_ses_con_TARGET"], ascending=[False, False]).head(1)
else:
    # Entre los que cumplen estabilidad, elegir el de mayor pe*fc (más demanda atendida),
    # con desempate por mayor margen p05 (más robusto).
    candidatos = candidatos.copy()
    candidatos["demanda_rel"] = candidatos["pe"]*candidatos["fc"]
    ganador = candidatos.sort_values(["demanda_rel","Margen_ses_p05"], ascending=[False, False]).head(1)

pe_sel = float(ganador["pe"].iloc[0])
fc_sel = float(ganador["fc"].iloc[0])

# 7) Cálculo detallado del escenario ganador (con EMS base)
ses_pico_sel = sessions_peak(Nm, Nmt, pe_sel, fc_sel)
carg_sel, cap_carg_sel = chargers_needed(ses_pico_sel, TS_ALG_MIN, U_ALG, SOCKETS_PER_CHARGER, HOURS_PEAK)
tomas_sel = carg_sel * SOCKETS_PER_CHARGER
kw_mean_per_socket = mean_socket_kw(Nm, Nmt, POWER_MOTO_KW, POWER_MOTOTAXI_KW)
pot_media_kw = tomas_sel * kw_mean_per_socket
pot_conserv_kw = tomas_sel * max(POWER_MOTO_KW, POWER_MOTOTAXI_KW)

# Contexto diario
arr_motos_day = Nm / PEAK_SHARE_DAY
arr_mtax_day  = Nmt / PEAK_SHARE_DAY
cargas_dia = (arr_motos_day + arr_mtax_day) * pe_sel * fc_sel
ts_h = TS_ALG_MIN/60.0
energia_dia_kwh = (arr_motos_day*pe_sel*fc_sel)*(POWER_MOTO_KW*ts_h) + \
                  (arr_mtax_day*pe_sel*fc_sel)*(POWER_MOTOTAXI_KW*ts_h)

df_recom = pd.DataFrame([{
    "pe_sel": pe_sel, "fc_sel": fc_sel,
    "Sesiones_pico_4h": ses_pico_sel,
    "Capacidad_por_cargador_ses_4h": cap_carg_sel,
    "Cargadores_requeridos": carg_sel,
    "Tomas_totales": tomas_sel,
    "Potencia_media_kW": pot_media_kw,
    "Potencia_conservadora_kW": pot_conserv_kw,
    "Llegadas_dia_total": arr_motos_day + arr_mtax_day,
    "Cargas_dia_total": cargas_dia,
    "Energia_dia_kWh": energia_dia_kwh
}])

# 8) Mostrar resultados
print("=== Escenario recomendado (automático) ===")
display(df_recom.style.format({
    "pe_sel": "{:.2%}", "fc_sel": "{:.2%}",
    "Sesiones_pico_4h": "{:.1f}",
    "Capacidad_por_cargador_ses_4h": "{:.1f}",
    "Cargadores_requeridos": "{:.0f}", "Tomas_totales": "{:.0f}",
    "Potencia_media_kW": "{:.1f}", "Potencia_conservadora_kW": "{:.1f}",
    "Llegadas_dia_total": "{:.0f}", "Cargas_dia_total": "{:.0f}",
    "Energia_dia_kWh": "{:.1f}"
}))

print("\n=== Grilla de escenarios (base EMS) ===")
display(df_grid.sort_values(["Cargadores_base","pe","fc"]).reset_index(drop=True).head(12))

print("\n=== Robustez (probabilidad de operar con 3 cargadores) ===")
display(df_rob.sort_values(["Prob_3_cargadores","pe","fc"], ascending=[False, True, True]).reset_index(drop=True).head(12))

# 9) Guardar a Excel y descargar
out_path = "/content/Seleccion_pe_fc_y_Dimensionamiento.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as xls:
    df_recom.to_excel(xls, sheet_name="Recomendado", index=False)
    df_grid.to_excel(xls, sheet_name="Escenarios_base", index=False)
    df_rob.to_excel(xls, sheet_name="Robustez_MC", index=False)
    df_merge.to_excel(xls, sheet_name="Escenarios_Resumen", index=False)

print("\nArchivo Excel creado en:", out_path)
try:
    from google.colab import files
    files.download(out_path)
except Exception as e:
    print("Si estás fuera de Colab, descarga el archivo desde el path indicado.")


=== Escenario recomendado (automático) ===


,pe_sel,fc_sel,Sesiones_pico_4h,Capacidad_por_cargador_ses_4h,Cargadores_requeridos,Tomas_totales,Potencia_media_kW,Potencia_conservadora_kW,Llegadas_dia_total,Cargas_dia_total,Energia_dia_kWh
0,15.00%,60.00%,92.7,35.3,3,12,25.5,36.0,1471,132,117.3



=== Grilla de escenarios (base EMS) ===


,pe,fc,Sesiones_pico_4h,Capacidad_por_cargador_ses_4h,Cargadores_base,Headroom_ses_con_TARGET
0,0.10,0.4,41.20,35.328,2,64.784
1,0.10,0.5,51.50,35.328,2,54.484
2,0.10,0.6,61.80,35.328,2,44.184
3,0.15,0.4,61.80,35.328,2,44.184
4,0.10,0.7,72.10,35.328,3,33.884
5,0.15,0.5,77.25,35.328,3,28.734
6,0.15,0.6,92.70,35.328,3,13.284
7,0.20,0.4,82.40,35.328,3,23.584
8,0.20,0.5,103.00,35.328,3,2.984
9,0.25,0.4,103.00,35.328,3,2.984



=== Robustez (probabilidad de operar con 3 cargadores) ===


,pe,fc,Prob_3_cargadores,Margen_ses_p05,Margen_ses_p50,Margen_ses_p95
0,0.10,0.4,1.000,53.407716,63.912072,77.455271
1,0.10,0.5,1.000,43.767814,55.139971,67.810709
2,0.10,0.6,1.000,33.048440,44.091749,58.081430
3,0.10,0.7,1.000,23.207576,33.789247,46.978816
4,0.15,0.4,1.000,33.521133,44.810711,58.020004
5,0.15,0.5,1.000,17.552362,28.366897,41.561024
6,0.20,0.4,1.000,12.288240,23.565277,37.762114
7,0.15,0.6,0.998,3.062467,13.996434,26.817429
8,0.20,0.5,0.632,-7.742678,3.551804,16.090538
9,0.25,0.4,0.628,-8.184354,2.959008,15.990306



Archivo Excel creado en: /content/Seleccion_pe_fc_y_Dimensionamiento.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>